# 1 — models and messages

`hotelbot` is a hotel-booking assistant built on `langchain`. Before it does
anything agent-like, it needs the one thing every book after this reuses: a
model call.

The `anthropic` SDK already does that — you saw it in the shop-assistant
project. What LangChain adds on top is a shared interface: the same
`invoke` / `stream` calls, the same message types, work no matter which
provider answers them. That matters starting book 2, when tool calls, and
later the whole agent loop, need to look the same whether the model behind
them is Anthropic or OpenAI.

This notebook does the smallest possible version of that: one call, read
what comes back, watch it stream, and see it fail to know things it was
never told. No tools, no agent yet — the model answers from the system
prompt alone.

In [1]:
from hotelbot.config import CHAT_MODEL, configure_tracing
from langchain.chat_models import init_chat_model

configure_tracing(book=1)

# reasoning_effort="low" keeps .content a plain string for this notebook —
# without it, Claude Sonnet 5 sometimes reasons before answering and .content
# becomes a list of blocks instead. That's a book-7-and-later topic.
model = init_chat_model(CHAT_MODEL, reasoning_effort="low")
model


ChatAnthropic(metadata={'lc_versions': {'langchain-core': '1.6.3', 'langchain': '1.4.0', 'langchain-anthropic': '1.7.2'}}, profile={'name': 'Claude Sonnet 5', 'release_date': '2026-06-29', 'last_updated': '2026-06-30', 'open_weights': False, 'max_input_tokens': 1000000, 'max_output_tokens': 128000, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': False, 'pdf_inputs': True, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': True, 'temperature': False, 'image_url_inputs': True, 'pdf_tool_message': True, 'image_tool_message': True, 'tool_call_streaming': True, 'reasoning_effort_levels': ['low', 'medium', 'high', 'xhigh', 'max'], 'reasoning_effort_default': 'high'}, model='claude-sonnet-5', max_tokens=128000, anthropic_api_url='https://api.anthropic.com', anthropic_api_key=SecretStr('**********'), model_kwargs={}, reasoning_eff

**One call.** `model.invoke(...)` takes a string (or a list of messages —
more on that next) and returns a single `AIMessage`. Three things on it are
worth looking at every time: `.content` is the text, `.response_metadata`
is what the provider reported about the call (stop reason, token usage,
which exact model answered), and `.id` is the id of that specific response.

Guess before running: how many output tokens does "Name three cities in
Portugal." cost?

In [2]:
response = model.invoke("Name three cities in Portugal.")

print(response.content)
print()
print("stop reason:", response.response_metadata["stop_reason"])
print("tokens:", response.usage_metadata)
print("id:", response.id)


Here are three cities in Portugal:

1. **Lisbon** – the capital and largest city
2. **Porto** – known for its wine production and historic riverside district
3. **Braga** – one of the oldest cities in Portugal, known for its religious heritage

stop reason: end_turn
tokens: {'input_tokens': 17, 'output_tokens': 85, 'total_tokens': 102, 'input_token_details': {'cache_read': 0, 'cache_creation': 0, 'ephemeral_5m_input_tokens': 0, 'ephemeral_1h_input_tokens': 0}, 'output_token_details': {'reasoning': 0}}
id: lc_run--01a0a8ef-7957-7550-998a-ca12221e7ce3-0


If tracing is on (`configure_tracing` printed nothing above, rather than
warning about a missing key), this call has a trace in LangSmith under the
`hotelbot-book-1` project — open it and match the token counts above
against what LangSmith recorded.

**Messages have roles.** A plain string sent to `.invoke` is shorthand for
a one-message list with an implicit human role. LangChain spells roles out
as classes: `SystemMessage` sets how the model should behave for the whole
conversation, `HumanMessage` is a turn from the user. Passing a list of
these is the real interface — the string shorthand just skips straight to
`HumanMessage`.

Ask the same question with and without a system prompt that tells the
model what it is, and compare the shape of the answer, not just the
words.

In [3]:
from langchain_core.messages import SystemMessage, HumanMessage

question = "I want a place to stay for a few days. What do you need to know?"

bare = model.invoke([HumanMessage(question)])
print("--- no system prompt ---")
print(bare.content)


--- no system prompt ---
Here's what would help me give you useful suggestions:

**Basics**
- Destination (city/region, or are you still deciding where to go?)
- Travel dates or approximate timing, and how many nights
- Number of guests, including any kids
- Budget range (per night or total)

**Preferences**
- Type of place: hotel, Airbnb/vacation rental, hostel, B&B, resort, etc.
- Location priorities: downtown/walkable, near specific attractions, quiet/residential, beach/nature, etc.
- Must-haves: kitchen, parking, pool, pet-friendly, accessibility needs, workspace, etc.
- Vibe: budget-friendly, mid-range comfort, or splurge/luxury

**Purpose of trip**
- Business, leisure, family visit, romantic getaway, solo trip, etc. — this affects what matters most (e.g., quiet vs. nightlife nearby)

If you just give me the destination and dates, I can start there and we can narrow down from your answers to the rest. What have you got so far?


In [4]:
system_prompt = (
    "You are a hotel booking assistant for Lisbon, Porto, Madrid, and "
    "Seville. Ask only for what you need to search: city, dates, and "
    "budget. Keep replies short."
)

briefed = model.invoke([SystemMessage(system_prompt), HumanMessage(question)])
print("--- with system prompt ---")
print(briefed.content)


--- with system prompt ---
Just a few details:

1. Which city — Lisbon, Porto, Madrid, or Seville?
2. Check-in and check-out dates?
3. Your budget per night?


Same question, same model — the system prompt is the only thing that
changed, and it changed what the model treats as its job: an open-ended
assistant hedges and lists general considerations, a briefed one asks for
city, dates, budget.

**The model can't see your data.** `hotelbot` is going to answer from a
real catalogue of hotels — but that catalogue lives in a JSON file the
model has never read. Ask it for a specific hotel now, before any tool
exists to look one up, and see what it does instead.

Predict first: will it refuse, hedge and ask for more detail, or just
invent a hotel?

In [5]:
ask = model.invoke([
    SystemMessage(system_prompt),
    HumanMessage("Recommend a hotel in Alfama, Lisbon, under €150 a night."),
])
print(ask.content)


Sure! Quick check first: what dates are you looking at, and how many guests?


It answers fluently — a name, a price, a vibe — and none of it is checked
against anything real. Compare against the one hotel record `hotelbot`
actually has:

In [6]:
from hotelbot.models import Hotel

known_hotel = Hotel(
    id="lis-004",
    name="Casa do Castelo",
    city="Lisbon",
    district="Alfama",
    price_per_night=140.0,
    rating=4.6,
    amenities=["breakfast", "wifi", "gym"],
    available=[["2026-10-01", "2026-10-31"], ["2026-12-01", "2026-12-20"]],
)
known_hotel


Hotel(id='lis-004', name='Casa do Castelo', city='Lisbon', district='Alfama', price_per_night=140.0, rating=4.6, amenities=['breakfast', 'wifi', 'gym'], available=[['2026-10-01', '2026-10-31'], ['2026-12-01', '2026-12-20']])

The model's answer and this record have nothing to do with each other —
whatever it named, it made up on the spot. `Hotel` is the shape every real
record in `data/hotels.json` will have from book 2 on; today it's just
proof that the model is answering blind. Closing that gap — giving the
model a real way to look up `known_hotel` instead of inventing one — is
what tools do, starting next book.

**A conversation is a list.** The model has no memory between calls — every
`.invoke` starts fresh. What looks like "remembering" is just resending
the whole conversation so far, including the model's own previous reply,
as the messages list. Build that list by hand: append the `AIMessage` you
got back, add a follow-up `HumanMessage`, and call again.

In [7]:
history = [SystemMessage(system_prompt), HumanMessage(question)]

turn_1 = model.invoke(history)
print(turn_1.content)


Just need a few details:

1. Which city — Lisbon, Porto, Madrid, or Seville?
2. Check-in and check-out dates?
3. Your budget per night?


`turn_1` goes back into the list as an `AIMessage`, then a new
`HumanMessage` — a real answer to the question the model itself just
asked — gets appended after it.

In [8]:
history.append(turn_1)
history.append(HumanMessage("Lisbon, October 10 to 12, up to 150 euros a night."))

turn_2 = model.invoke(history)
print(turn_2.content)


[{'signature': 'Eo8CCpABCBEYAipAjuKrIBYLqGWjtn1UPneEyC1yCogug+x0nJ9kqSoeodV1ZaPajAznNbqkSt/vKtK214/RD6QiWPZZwD1M0l2cmjIPY2xhdWRlLXNvbm5ldC01OABCCHRoaW5raW5nWiQ0Y2UxZTM5NC00ZjhiLTQ5MDItYjYzYi0yYTUwY2JjMDMzZmSoAcbyqNUGEgxn5PMYdzuAB8GMwv8aDMT8yb3JniUpKcE3AiIwWBUQFzo2KplLGcXZWSn6DBEHAGEhd9B26OHAPx2M4rC2Q6UtjsxCP5pDm517BcKuKixK28pQuqsqbUmoV1/Kh521xP+cSEAoMwAyYKiytlBqbBsWzr1e+gigrnXToRgB', 'thinking': 'That confirms it, so I can move forward.\n\n', 'type': 'thinking'}, {'text': 'Got it — Lisbon, Oct 10–12, up to €150/night, 2 nights.\n\nAny preferences on area or hotel type (e.g., central, boutique, family-friendly), or should I just search with these criteria?', 'type': 'text'}]


The model now answers as if it "remembers" the first message — it doesn't;
`history` just carries the whole exchange to it again. That has a cost:
guess before running whether `turn_2`'s `input_tokens` is bigger than
`turn_1`'s, and by roughly how much.

In [9]:
print("turn 1 input tokens:", turn_1.usage_metadata["input_tokens"])
print("turn 2 input tokens:", turn_2.usage_metadata["input_tokens"])


turn 1 input tokens: 85
turn 2 input tokens: 171


`turn_2` pays for every token in `turn_1`'s system prompt, question, and
answer, plus its own new message — the list only grows, and so does the
bill for resending it. That's the tradeoff book 5's summarization
middleware exists to manage; for now, just watch the number climb.

**Streaming.** `.invoke` waits for the full answer before returning
anything. `.stream` instead yields pieces as the model produces them, each
piece an `AIMessageChunk`. A chunk's `.content` is the raw delta — for a
streamed response that's a list of small block fragments, not plain text —
so read `.text` instead, which gives you the plain string either way.
Chunks also support `+`, which merges their text and metadata into one
combined message, so you can print incrementally and still end up with the
same kind of object `.invoke` would have given you.

In [10]:
full = None
for chunk in model.stream([HumanMessage("Name three cities in Spain, one per line.")]):
    print(chunk.text, end="", flush=True)
    full = chunk if full is None else full + chunk

print()
print()
print(type(full))
print(full.text)


Madrid
Barcelona
Valencia

<class 'langchain_core.messages.ai.AIMessageChunk'>
Madrid
Barcelona
Valencia


Nothing about the final answer changed — same model, same question,
same kind of message once the chunks are summed. `.stream` only changes
when you get to see the words: as they're produced, instead of all at
once.

**The provider is a string.** Everything above — `SystemMessage`,
`HumanMessage`, `.invoke`, `.stream`, `.text`, `usage_metadata` — is
LangChain's interface, not Anthropic's. Swap the model id's prefix from
`anthropic:` to `openai:` and the same messages, the same calls, the same
attributes all still work; only which provider answers changes.

In [11]:
openai_model = init_chat_model("openai:gpt-5-mini")

openai_response = openai_model.invoke([SystemMessage(system_prompt), HumanMessage(question)])
print(openai_response.content)


Which city — Lisbon, Porto, Madrid, or Seville? What are your check‑in and check‑out dates? What's your budget per night?


Same code, a different voice answering — that's the payoff of calling
through LangChain instead of each provider's own SDK directly. The rest
of this project stays on Anthropic; `hotelbot`'s `CHAT_MODEL` stays
`anthropic:claude-sonnet-5` from here on.

You can now call a chat model through LangChain, read an `AIMessage`'s
content and metadata, build up a conversation by resending it as a
growing list, stream a response chunk by chunk, and point the same code
at a different provider by changing one string.

What it still can't do: know anything that wasn't typed into the prompt.
It invented a hotel in cell 4 because nothing gave it a real one to check
against. Book 2 gives it hands — tools it can call to look up real data —
starting with the same `search_hotels` and `get_hotel` functions this
notebook's `Hotel` model was previewing.